In [2]:
# ============================================================
#   REVENUE PREDICTION MODEL — FINAL VERSION
#   Place Final_Dataset_ModelReady.csv in same folder
#   Run each cell in order in Jupyter Notebook
#
#   SPLIT:
#     TRAIN   → Q2 2023 to Q4 2024
#     TEST    → Q1 2025 to Q4 2025  (verify accuracy)
#     PREDICT → Q1 2026             (real prediction)
#
#   MODELS:
#     Large cap → ElasticNet
#     Mid cap   → ElasticNet
#     Small cap → XGBoost (KNR removed, exceptional flags added)
# ============================================================


# ════════════════════════════════════════════════════════════
# CELL 1 — Install packages (run once, then restart kernel)
# ════════════════════════════════════════════════════════════

# !pip install xgboost scikit-learn pandas numpy joblib


# ════════════════════════════════════════════════════════════
# CELL 2 — Imports
# ════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit

print("✅ All packages imported")


# ════════════════════════════════════════════════════════════
# CELL 3 — Fix CSV: add exceptional flags + remove KNR
# ════════════════════════════════════════════════════════════

raw = pd.read_csv('Final_Dataset_ModelReady.csv', header=1)
print(f"Original: {len(raw)} rows")

# Add is_exceptional flags for known one-time shock quarters
exceptional_rows = [
    ('UltraTech Cement',       2025, 'Q2'),  # merger impact
    ('Balaji Amines',          2025, 'Q1'),  # chemical cycle bottom
    ('Cera Sanitaryware',      2025, 'Q2'),  # demand shock
    ('Capacite Infraprojects', 2025, 'Q1'),  # project payment delay
]
for company, year, quarter in exceptional_rows:
    mask = (
        (raw['Company'] == company) &
        (raw['Year']    == year)    &
        (raw['Quarter'] == quarter)
    )
    raw.loc[mask, 'is_exceptional'] = 1
    print(f"  ✅ Flagged {company} {quarter} {year} as exceptional")

# Remove KNR Constructions — too volatile to predict (govt payment timing)
raw = raw[raw['Company'] != 'KNR Constructions'].copy()
print(f"  ✅ Removed KNR Constructions")
print(f"Final: {len(raw)} rows")

# Save cleaned version
raw.to_csv('Final_Dataset_ModelReady_v2.csv', index=False)
print("✅ Saved as Final_Dataset_ModelReady_v2.csv")


# ════════════════════════════════════════════════════════════
# CELL 4 — Load and prepare data
# ════════════════════════════════════════════════════════════

df = pd.read_csv('Final_Dataset_ModelReady_v2.csv')
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

df = df.sort_values(['Company', 'Year', 'Quarter_Type_Encoded']).reset_index(drop=True)

# Drop first-quarter rows (lag features are NaN — can't train on these)
df = df[df['_is_first_quarter'] == False].copy()
# Keep _drop_no_target rows (Q4 2025) — needed to predict Q1 2026
df = df.reset_index(drop=True)
print(f"After dropping first-quarter rows: {df.shape[0]} rows")

# Drop leakage + redundant columns
DROP_COLS = [
    'Revenue', 'Profit', 'EBITDA', 'Stock_Price',
    'Profit_Margin', 'EBITDA_Margin', 'Profit_to_Ebitda',
    'Revenue_QoQ', 'Profit_QoQ',
    'Quarters_elapsed',
    'Repo_rate_cat',
    '_is_first_quarter',
    '_drop_no_target',
]
df.drop(columns=DROP_COLS, inplace=True)

# Target: next quarter revenue
df['Target_Revenue'] = df.groupby('Company')['Revenue_Lag1'].shift(-1)

# Sector-specific macro signals
usd_sectors       = ['IT', 'Pharma', 'Auto Ancillary', 'Textiles']
crude_sectors     = ['Energy', 'Chemicals', 'Plastics']
inflation_sectors = ['FMCG', 'Retail', 'Banking', 'Consumer',
                     'Cement', 'Infrastructure', 'Building Materials']

df['USD_INR_signal']   = df['USD_INR']         * df['Sector'].isin(usd_sectors).astype(int)
df['Crude_signal']     = df['Crude_oil_price']  * df['Sector'].isin(crude_sectors).astype(int)
df['Inflation_signal'] = df['Inflation']        * df['Sector'].isin(inflation_sectors).astype(int)

# Cap-type interaction features
df['RevLag_x_Cap']  = df['Revenue_Lag1']        * df['Cap_Type_Encoded']
df['ProfLag_x_Cap'] = df['Profit_Lag1']          * df['Cap_Type_Encoded']
df['Accel_x_Cap']   = df['Revenue_acceleration'] * df['Cap_Type_Encoded']
df['Moment_x_Cap']  = df['Price_momentum']       * df['Cap_Type_Encoded']

print(f"Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print("✅ Data preparation complete")


# ════════════════════════════════════════════════════════════
# CELL 5 — Define features
# ════════════════════════════════════════════════════════════

FEATURES = [
    'Cap_Type_Encoded',
    'Quarter_Type_Encoded',
    'Company_TargetEncoded',
    'Sector_TargetEncoded',
    'is_Q4',
    'is_exceptional',
    'USD_INR_signal',
    'Crude_signal',
    'Inflation_signal',
    'Revenue_Lag1',
    'Profit_Lag1',
    'Stock_Lag1',
    'EBITDA_Lag1',
    'Revenue_acceleration',
    'Price_momentum',
    'RevLag_x_Cap',
    'ProfLag_x_Cap',
    'Accel_x_Cap',
    'Moment_x_Cap',
]

print(f"Total features: {len(FEATURES)}")
for i, f in enumerate(FEATURES, 1):
    print(f"  {i:2d}. {f}")


# ════════════════════════════════════════════════════════════
# CELL 6 — Train / Test / Predict split
# ════════════════════════════════════════════════════════════

df_known   = df[df['Target_Revenue'].notna()].copy()
df_predict = df[df['Target_Revenue'].isna()].copy()

train_df = df_known[df_known['Year'] <= 2024].copy()
test_df  = df_known[df_known['Year'] == 2025].copy()

print(f"TRAIN   : {len(train_df)} rows → 2023 Q3 to 2024 Q4")
print(f"TEST    : {len(test_df)} rows → 2025 Q1 to Q4 (actual answers known ✅)")
print(f"PREDICT : {len(df_predict)} rows → Q4 2025 (predicting Q1 2026)")


# ════════════════════════════════════════════════════════════
# CELL 7 — Train 3 models
# ════════════════════════════════════════════════════════════

joblib.dump(best_model, 'models/Large_model.pkl')
os.makedirs('models', exist_ok=True)
TSCV        = TimeSeriesSplit(n_splits=3)
results     = {}
model_names = {'Large': 'ElasticNet', 'Mid': 'ElasticNet', 'Small': 'XGBoost'}

for cap in ['Large', 'Mid', 'Small']:

    print(f"\n{'='*55}")
    print(f"  {cap} cap — {model_names[cap]}")
    print(f"{'='*55}")

    tr = train_df[train_df['Cap_Type'] == cap].copy()
    te = test_df[ test_df['Cap_Type']  == cap].copy()

    # Target encoding from TRAIN only — no leakage
    company_mean = tr.groupby('Company')['Revenue_Lag1'].mean()
    sector_mean  = tr.groupby('Sector')['Revenue_Lag1'].mean()
    for d in [tr, te]:
        d['Company_TargetEncoded'] = d['Company'].map(company_mean)\
                                       .fillna(company_mean.mean())
        d['Sector_TargetEncoded']  = d['Sector'].map(sector_mean)\
                                       .fillna(sector_mean.mean())

    X_train = tr[FEATURES].fillna(0).values
    y_train = tr['Target_Revenue'].values
    X_test  = te[FEATURES].fillna(0).values
    y_test  = te['Target_Revenue'].values

    print(f"  Train: {len(X_train)} rows  |  Test: {len(X_test)} rows")

    # ── ElasticNet for Large and Mid ──────────────────────────
    if cap in ['Large', 'Mid']:
        scaler    = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)

        best_model, best_score = None, -999
        for alpha in [0.01, 0.1, 1, 10, 100]:
            for l1 in [0.2, 0.5, 0.8]:
                en = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=10000)
                fold_scores = []
                for tr_i, val_i in TSCV.split(X_train_s):
                    en.fit(X_train_s[tr_i], y_train[tr_i])
                    fold_scores.append(r2_score(
                        y_train[val_i], en.predict(X_train_s[val_i])))
                if np.mean(fold_scores) > best_score:
                    best_score = np.mean(fold_scores)
                    best_model = ElasticNet(
                        alpha=alpha, l1_ratio=l1, max_iter=10000)

        best_model.fit(X_train_s, y_train)
        train_preds = best_model.predict(X_train_s)
        test_preds  = best_model.predict(X_test_s)

        te_copy = te.copy()
        te_copy['pred'] = test_preds
        company_errors  = {
            comp: mean_absolute_percentage_error(
                grp['Target_Revenue'], grp['pred'])
            for comp, grp in te_copy.groupby('Company')
        }

        joblib.dump({
            'model'         : best_model,
            'scaler'        : scaler,
            'company_mean'  : company_mean,
            'sector_mean'   : sector_mean,
            'company_errors': company_errors,
            'global_mape'   : mean_absolute_percentage_error(y_test, test_preds),
            'model_name'    : 'ElasticNet',
        }, f'models/{cap}_model.pkl')

    # ── XGBoost for Small ─────────────────────────────────────
    else:
        xgb = XGBRegressor(
            n_estimators     = 200,
            learning_rate    = 0.05,
            max_depth        = 3,
            min_child_weight = 5,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            reg_alpha        = 0.1,
            reg_lambda       = 1.0,
            random_state     = 42,
            verbosity        = 0
        )
        xgb.fit(X_train, y_train)
        train_preds = xgb.predict(X_train)
        test_preds  = xgb.predict(X_test)

        te_copy = te.copy()
        te_copy['pred'] = test_preds
        company_errors  = {
            comp: mean_absolute_percentage_error(
                grp['Target_Revenue'], grp['pred'])
            for comp, grp in te_copy.groupby('Company')
        }

        joblib.dump({
            'model'         : xgb,
            'scaler'        : None,
            'company_mean'  : company_mean,
            'sector_mean'   : sector_mean,
            'company_errors': company_errors,
            'global_mape'   : mean_absolute_percentage_error(y_test, test_preds),
            'model_name'    : 'XGBoost',
        }, f'models/{cap}_model.pkl')

    train_r2  = r2_score(y_train, train_preds)
    test_r2   = r2_score(y_test,  test_preds)
    test_mape = mean_absolute_percentage_error(y_test, test_preds) * 100

    print(f"  Train R²   : {train_r2:.4f}")
    print(f"  Test  R²   : {test_r2:.4f}")
    print(f"  Test  MAPE : {test_mape:.2f}%")
    results[cap] = dict(train_r2=train_r2, test_r2=test_r2, test_mape=test_mape)

# Save assets for prediction
joblib.dump(df,       'models/full_data.pkl')
joblib.dump(FEATURES, 'models/features.pkl')
company_info = df[['Company', 'Sector', 'Cap_Type', 'Cap_Type_Encoded']]\
    .drop_duplicates().set_index('Company')
joblib.dump(company_info, 'models/company_info.pkl')

print(f"\n{'='*60}")
print("  FINAL RESULTS")
print(f"{'='*60}")
print(f"  {'Cap':<8} {'Model':<12} {'Train R²':>10} {'Test R²':>10} {'MAPE':>8} {'Gap':>8}  {'Status'}")
print(f"  {'-'*60}")
for cap, r in results.items():
    gap    = r['train_r2'] - r['test_r2']
    status = "✅ Healthy" if gap < 0.05 else ("⚠️  Mild" if gap < 0.15 else "❌ Overfit")
    print(f"  {cap:<8} {model_names[cap]:<12} "
          f"{r['train_r2']:>10.4f} {r['test_r2']:>10.4f} "
          f"{r['test_mape']:>7.2f}% {gap:>8.4f}  {status}")
print("\n✅ All models saved to models/ folder")


# ════════════════════════════════════════════════════════════
# CELL 8 — Predicted vs Actual on 2025 test data
# ════════════════════════════════════════════════════════════

print("\n" + "="*75)
print("  PREDICTED vs ACTUAL — 2025 TEST SET")
print("  ✅ error < 10%   ⚠ error < 20%   ❌ error > 20%")
print("="*75)
print(f"  {'Company':<25} {'Qtr':<6} {'Actual':>14} {'Predicted':>14} {'Error %':>8}")
print("  " + "-"*70)

for cap in ['Large', 'Mid', 'Small']:
    bundle = joblib.load(f'models/{cap}_model.pkl')
    model, scaler  = bundle['model'], bundle['scaler']
    c_mean, s_mean = bundle['company_mean'], bundle['sector_mean']

    te = test_df[test_df['Cap_Type'] == cap].copy()
    te['Company_TargetEncoded'] = te['Company'].map(c_mean).fillna(c_mean.mean())
    te['Sector_TargetEncoded']  = te['Sector'].map(s_mean).fillna(s_mean.mean())

    X = te[FEATURES].fillna(0).values
    if scaler: X = scaler.transform(X)
    te = te.copy()
    te['Predicted'] = model.predict(X)

    for _, row in te.sort_values(['Company', 'Quarter_Type_Encoded']).iterrows():
        err  = abs(row['Target_Revenue'] - row['Predicted']) / row['Target_Revenue'] * 100
        flag = " ✅" if err < 10 else (" ⚠" if err < 20 else " ❌")
        print(f"  {row['Company']:<25} {row['Quarter']:<6} "
              f"{row['Target_Revenue']:>14,.1f} {row['Predicted']:>14,.1f} "
              f"{err:>7.1f}%{flag}")


# ════════════════════════════════════════════════════════════
# CELL 9 — Overfitting check
# ════════════════════════════════════════════════════════════

print("\n" + "="*65)
print("  OVERFITTING CHECK")
print("="*65)
print(f"  {'Cap':<8} {'Model':<12} {'Train R²':>10} {'Test R²':>10} {'Gap':>8}  {'Status'}")
print(f"  {'-'*60}")
for cap, r in results.items():
    gap    = r['train_r2'] - r['test_r2']
    status = "✅ Healthy" if gap < 0.05 else ("⚠️  Mild" if gap < 0.15 else "❌ Overfit!")
    print(f"  {cap:<8} {model_names[cap]:<12} "
          f"{r['train_r2']:>10.4f} {r['test_r2']:>10.4f} "
          f"{gap:>8.4f}  {status}")
print("\n  Rule: Gap < 0.05 = Healthy | 0.05–0.15 = Mild | >0.15 = Overfit")


# ════════════════════════════════════════════════════════════
# CELL 10 — Prediction function
# ════════════════════════════════════════════════════════════

QUARTER_MAP = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}
QUARTER_REV = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}

def predict_revenue(company_name, target_quarter, target_year):
    """
    Predict next quarter revenue for any company.

    Parameters:
        company_name   : e.g. 'Trent'
        target_quarter : 'Q1', 'Q2', 'Q3', or 'Q4'
        target_year    : e.g. 2026

    Example:
        predict_revenue('Trent', 'Q1', 2026)
        predict_revenue('TCS',   'Q1', 2026)
    """
    company_info = joblib.load('models/company_info.pkl')
    full_df      = joblib.load('models/full_data.pkl')
    FEATURES     = joblib.load('models/features.pkl')

    # Validate company
    if company_name not in company_info.index:
        close = [c for c in company_info.index
                 if company_name.lower() in c.lower()]
        print(f"\n❌ '{company_name}' not found.")
        if close:
            print(f"   Did you mean: {close}")
        else:
            print("   Available companies:")
            for c in sorted(company_info.index):
                print(f"     • {c}")
        return None

    cap_type = company_info.loc[company_name, 'Cap_Type']
    cap_enc  = company_info.loc[company_name, 'Cap_Type_Encoded']
    sector   = company_info.loc[company_name, 'Sector']

    bundle         = joblib.load(f'models/{cap_type}_model.pkl')
    model          = bundle['model']
    scaler         = bundle['scaler']
    company_mean   = bundle['company_mean']
    sector_mean    = bundle['sector_mean']
    company_errors = bundle['company_errors']
    global_mape    = bundle['global_mape']
    model_name     = bundle['model_name']

    # Find lag row — one quarter before target
    tq_num   = QUARTER_MAP[target_quarter.upper()]
    lag_q    = 4 if tq_num == 1 else tq_num - 1
    lag_year = target_year - 1 if tq_num == 1 else target_year

    comp_df = full_df[full_df['Company'] == company_name]
    lag_row = comp_df[
        (comp_df['Year']                 == lag_year) &
        (comp_df['Quarter_Type_Encoded'] == lag_q)
    ]

    if lag_row.empty:
        print(f"\n❌ No data for {company_name} in "
              f"{QUARTER_REV[lag_q]} {lag_year}.")
        print(f"   Available: {comp_df[['Year','Quarter']].values.tolist()}")
        return None

    lag    = lag_row.iloc[0]
    latest = full_df.sort_values(['Year', 'Quarter_Type_Encoded']).iloc[-1]

    usd_sectors       = ['IT', 'Pharma', 'Auto Ancillary', 'Textiles']
    crude_sectors     = ['Energy', 'Chemicals', 'Plastics']
    inflation_sectors = ['FMCG', 'Retail', 'Banking', 'Consumer',
                         'Cement', 'Infrastructure', 'Building Materials']

    usd_sig   = float(latest['USD_INR'])         if sector in usd_sectors       else 0.0
    crude_sig = float(latest['Crude_oil_price'])  if sector in crude_sectors     else 0.0
    inf_sig   = float(latest['Inflation'])        if sector in inflation_sectors else 0.0

    rev_lag1  = float(lag['Revenue_Lag1']         or 0)
    prof_lag1 = float(lag['Profit_Lag1']          or 0)
    accel     = float(lag['Revenue_acceleration'] or 0)
    momentum  = float(lag['Price_momentum']       or 0)

    row = {
        'Cap_Type_Encoded'     : cap_enc,
        'Quarter_Type_Encoded' : tq_num,
        'Company_TargetEncoded': company_mean.get(company_name, company_mean.mean()),
        'Sector_TargetEncoded' : sector_mean.get(sector, sector_mean.mean()),
        'is_Q4'                : 1 if tq_num == 4 else 0,
        'is_exceptional'       : 0,
        'USD_INR_signal'       : usd_sig,
        'Crude_signal'         : crude_sig,
        'Inflation_signal'     : inf_sig,
        'Revenue_Lag1'         : rev_lag1,
        'Profit_Lag1'          : prof_lag1,
        'Stock_Lag1'           : float(lag['Stock_Lag1']  or 0),
        'EBITDA_Lag1'          : float(lag['EBITDA_Lag1'] or 0),
        'Revenue_acceleration' : accel,
        'Price_momentum'       : momentum,
        'RevLag_x_Cap'         : rev_lag1  * cap_enc,
        'ProfLag_x_Cap'        : prof_lag1 * cap_enc,
        'Accel_x_Cap'          : accel     * cap_enc,
        'Moment_x_Cap'         : momentum  * cap_enc,
    }

    X = np.array([[row[f] for f in FEATURES]])
    if scaler: X = scaler.transform(X)

    pred      = float(model.predict(X)[0])
    mape_used = company_errors.get(company_name, global_mape)
    lower     = pred * (1 - mape_used)
    upper     = pred * (1 + mape_used)

    macro_used = []
    if usd_sig:   macro_used.append('USD/INR')
    if crude_sig: macro_used.append('Crude oil')
    if inf_sig:   macro_used.append('Inflation')
    if not macro_used: macro_used = ['None (domestic company)']

    print(f"\n{'='*55}")
    print(f"  REVENUE PREDICTION")
    print(f"{'='*55}")
    print(f"  Company         : {company_name}")
    print(f"  Predicting      : {target_quarter.upper()} {target_year}")
    print(f"  Cap Type        : {cap_type}  |  Model : {model_name}")
    print(f"  Sector          : {sector}")
    print(f"  Macro signal    : {', '.join(macro_used)}")
    print(f"  Using lag data  : {QUARTER_REV[lag_q]} {lag_year}  "
          f"(Revenue = ₹{rev_lag1:,.0f} Cr)")
    print(f"{'─'*55}")
    print(f"  Predicted Revenue  :  ₹{pred:>12,.1f} Cr")
    print(f"  Confidence Range   :  ₹{lower:>12,.1f} Cr")
    print(f"                     –  ₹{upper:>12,.1f} Cr")
    print(f"  Model accuracy     :  ±{mape_used*100:.1f}% (MAPE on 2025 test)")
    print(f"{'='*55}\n")

    return {
        'company'          : company_name,
        'quarter'          : f"{target_quarter.upper()} {target_year}",
        'predicted_revenue': round(pred, 2),
        'lower_bound'      : round(lower, 2),
        'upper_bound'      : round(upper, 2),
        'mape_pct'         : round(mape_used * 100, 2),
        'model_used'       : model_name,
        'cap_type'         : cap_type,
    }

print("✅ Prediction function ready")


# ════════════════════════════════════════════════════════════
# CELL 11 — Show all available companies (KNR removed)
# ════════════════════════════════════════════════════════════

company_info = joblib.load('models/company_info.pkl')
print(f"\n  {'#':<4} {'Company':<30} {'Sector':<22} {'Cap Type'}")
print(f"  {'-'*65}")
for i, comp in enumerate(sorted(company_info.index), 1):
    r = company_info.loc[comp]
    print(f"  {i:<4} {comp:<30} {r['Sector']:<22} {r['Cap_Type']}")
print(f"\n  Total: {len(company_info)} companies (KNR Constructions removed)")


# ════════════════════════════════════════════════════════════
# CELL 12 — Single prediction
# Change company, quarter, year below and run this cell
# ════════════════════════════════════════════════════════════

result = predict_revenue(
    company_name   = 'Trent',   # ← any company from list above
    target_quarter = 'Q1',      # ← Q1 / Q2 / Q3 / Q4
    target_year    = 2026       # ← 2026 = real prediction
                                #   2025 = verify against known data
)


# ════════════════════════════════════════════════════════════
# CELL 13 — Predict Q1 2026 for all 29 companies + save CSV
# ════════════════════════════════════════════════════════════

company_info = joblib.load('models/company_info.pkl')

print("\n" + "="*82)
print("  Q1 2026 REVENUE PREDICTIONS — ALL 29 COMPANIES")
print("="*82)
print(f"  {'Company':<28} {'Cap':<8} {'Model':<12} "
      f"{'Predicted (₹ Cr)':>16}  {'Confidence Range':<30}  {'±MAPE'}")
print("  " + "-"*82)

all_results = []
for company in sorted(company_info.index):
    r = predict_revenue(company, 'Q1', 2026)
    if r:
        all_results.append(r)
        print(f"  {r['company']:<28} {r['cap_type']:<8} {r['model_used']:<12} "
              f"{r['predicted_revenue']:>16,.1f}  "
              f"₹{r['lower_bound']:>10,.1f} – ₹{r['upper_bound']:<12,.1f}  "
              f"±{r['mape_pct']:.1f}%")

pd.DataFrame(all_results).to_csv('Q1_2026_Predictions.csv', index=False)
print(f"\n✅ Saved to Q1_2026_Predictions.csv")

✅ All packages imported
Original: 330 rows
  ✅ Flagged UltraTech Cement Q2 2025 as exceptional
  ✅ Flagged Balaji Amines Q1 2025 as exceptional
  ✅ Flagged Cera Sanitaryware Q2 2025 as exceptional
  ✅ Flagged Capacite Infraprojects Q1 2025 as exceptional
  ✅ Removed KNR Constructions
Final: 319 rows
✅ Saved as Final_Dataset_ModelReady_v2.csv
Loaded: 319 rows × 35 columns
After dropping first-quarter rows: 290 rows
Final dataset: 290 rows × 30 columns
✅ Data preparation complete
Total features: 19
   1. Cap_Type_Encoded
   2. Quarter_Type_Encoded
   3. Company_TargetEncoded
   4. Sector_TargetEncoded
   5. is_Q4
   6. is_exceptional
   7. USD_INR_signal
   8. Crude_signal
   9. Inflation_signal
  10. Revenue_Lag1
  11. Profit_Lag1
  12. Stock_Lag1
  13. EBITDA_Lag1
  14. Revenue_acceleration
  15. Price_momentum
  16. RevLag_x_Cap
  17. ProfLag_x_Cap
  18. Accel_x_Cap
  19. Moment_x_Cap
TRAIN   : 174 rows → 2023 Q3 to 2024 Q4
TEST    : 87 rows → 2025 Q1 to Q4 (actual answers known ✅)
PR

In [4]:
bundle = joblib.load('models/Large_model.pkl')